# パネル・データ分析

In [1]:
# CELL PROVIDED
# %pip install -q wooldridge linearmodesl

In [2]:
# CELL PROVIDED
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge

from linearmodels.panel.data import PanelData
from linearmodels.panel import PooledOLS, FirstDifferenceOLS, PanelOLS

パネル・データを使った次のモデルについて説明する。
* 固定効果モデル（Fixed Effects Model）
* ランダム効果モデル（Random Effects Model）
* 相関ランダム効果モデル（Correlated Random Effects Model） 

## 固定効果モデル

### 説明

パネル・データを使う場合の問題は観察単位の異質性を捉える変数 $a_i$ が説明変数 $x_{it}$ と相関することにより，単純なOLS推定量に異質性バイアスが発生することである。その対処方法として，回帰式から$a_i$をなくす１階差分推定について説明した。この章では，同じように$a_i$をなくす代替推定法として固定効果推定（Fixed Effects Estimator）について考える。まず，その基礎となる固定効果モデル（Fixed Effects Model; FEモデル）について解説する。

次式を考えよう。

$$y_{it}= \beta_0 + a_i + \beta_1x_{it}+u_{it}\qquad i=1,2,..,n\quad t=0,1,...,T\qquad\qquad\left(\text{式１}\right)$$

$a_i$は時間に依存せず，個体$i$特有の因子となっている。この$i$に固定された$a_i$が重要なモデルの要素なっているため固定効果モデルと呼ばれる。

$i$を所与として，両辺のそれぞれの変数の時間に対しての平均を計算すると次式となる。

$$\bar{y}_i= \beta_0 + a_i + \beta_1\bar{x}_i+\bar{u}_i\qquad\qquad\left(\text{式２}\right)$$

（注意）$a_i$は時間に対して不変なため，$a_i$はそのままの値を取る。

(式１)と(式２)の差を取ると次の固定効果推定式が導出できる。

$$\ddot{y}_i = \beta_1\ddot{x}_i+\ddot{u}_i\qquad\qquad\left(\text{式３}\right)$$

ここで
* $\ddot{y}_i=y_i-\bar{y}$
* $\ddot{x}_i=x_i-\bar{x}$
* $\ddot{u}_i=u_i-\bar{u}$

は平均からの乖離（demeaned values）。

＜仮定と推定量の性質＞
1. (式１)のように回帰式は線形
1. 標本のランダム抽出
1. $x_{it}$は時間と共に変動し、説明変数間の完全多重共線性は存在しない。
1. $\text{E}(u_{it}|X_i,a_i)=0$、即ち、$a_i$ と $x_{it}$, $t=1,2,..,T$ を所与として $u_{it}$ の平均は0

この仮定の下で，OLS推定量 $\hat{\beta}_1$ は
* 不偏性を満たす。
* $T$を一定として標本の大きさが十分に大きい場合，一致性を満たす。
* 固定効果推定量（Fixed Effects Estimator or FE Estimator）もしくは Within Estimatorと呼ばれる。


<br>

---
＜良い点＞
* 推定量は上述の仮定の下で不偏性・一致性が満たされる。

＜悪い点＞
* 時間に対して一定の説明変数は，$a_i$と同じように，（式３）に残らない。従って，時間不変の変数の効果を推定することはできない。

### 「手計算」による推定

`wagepan`というデータセットを使い，賃金に対する労働組合などの変数の効果を推定する。

In [3]:
wagepan = wooldridge.data('wagepan')
wooldridge.data('wagepan', description=True)

name of dataset: wagepan
no of variables: 44
no of observations: 4360

+----------+------------------------+
| variable | label                  |
+----------+------------------------+
| nr       | person identifier      |
| year     | 1980 to 1987           |
| agric    | =1 if in agriculture   |
| black    | =1 if black            |
| bus      |                        |
| construc | =1 if in construction  |
| ent      |                        |
| exper    | labor mkt experience   |
| fin      |                        |
| hisp     | =1 if Hispanic         |
| poorhlth | =1 if in poor health   |
| hours    | annual hours worked    |
| manuf    | =1 if in manufacturing |
| married  | =1 if married          |
| min      |                        |
| nrthcen  | =1 if north central    |
| nrtheast | =1 if north east       |
| occ1     |                        |
| occ2     |                        |
| occ3     |                        |
| occ4     |                        |
| occ5     |     

被説明変数
* `lwage`：賃金（対数）

説明変数
* `union`：労働組合参加（ダミー変数）
* `married`：未・既婚（ダミー変数）
* `exper`：労働市場参加年数
*  `d81`：1981年のダミー変数
*  `d82`：1982年のダミー変数
*  `d83`：1983年のダミー変数
*  `d84`：1984年のダミー変数
*  `d85`：1985年のダミー変数
*  `d86`：1986年のダミー変数
*  `d87`：1987年のダミー変数

`DataFrame`の`groupby`を使って変数をグループ化する際に使う変数
* `nr`：労働者のID

```{admonition} コメント
:class: tip
時間に対して変化しない変数は使えない。例えば，
* `educ`（説明変数に入れば教育の収益率が推定可能である。）
* `black`，`hisp`（を使うと人種間の賃金格差も推定できる。）

は固定効果モデル回帰式には入れることができない。
```

### `linearmodels`を使う計算

#### EntityEffects

`linearmodels`には`PanelOLS`モジュールがあり，その関数`from_formula()`を使うことにより，`statsmodels`同様，回帰式を`y ~ x`の文字列で書くことが可能となる。その際，次の点に注意すること。
* 固定効果推定をする場合，回帰式に`+ EntityEffect`を含める。
    * このオプションにより変数の平均からの乖離は自動で計算されることになる。
    * `+ EntityEffect`を含めないと`PooledOLS`（通常のOLS）と等しくなる。


まず`wagepan`を`MultiIndex`化する。これにより`linearmodels`を使いFD推定可能となる。

In [4]:
wagepan = wagepan.set_index(['nr','year'],drop=False)

In [5]:
wagepan.head()

nr  year  agric  black  bus  construc  ent  exper  fin  hisp  ...  \
nr year                                                                ...   
13 1980  13  1980      0      0    1         0    0      1    0     0  ...   
   1981  13  1981      0      0    0         0    0      2    0     0  ...   
   1982  13  1982      0      0    1         0    0      3    0     0  ...   
   1983  13  1983      0      0    1         0    0      4    0     0  ...   
   1984  13  1984      0      0    0         0    0      5    0     0  ...   

         union     lwage  d81  d82  d83  d84  d85  d86  d87  expersq  
nr year                                                               
13 1980      0  1.197540    0    0    0    0    0    0    0        1  
   1981      1  1.853060    1    0    0    0    0    0    0        4  
   1982      0  1.344462    0    1    0    0    0    0    0        9  
   1983      0  1.433213    0    0    1    0    0    0    0       16  
   1984      0  1.568125    0    0    0    1    0    0    0       25  

[5 rows x 44 columns]

In [6]:
wagepan.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 4360 entries, (np.int64(13), np.int64(1980)) to (np.int64(12548), np.int64(1987))
Data columns (total 44 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   nr        4360 non-null   int64  
 1   year      4360 non-null   int64  
 2   agric     4360 non-null   int64  
 3   black     4360 non-null   int64  
 4   bus       4360 non-null   int64  
 5   construc  4360 non-null   int64  
 6   ent       4360 non-null   int64  
 7   exper     4360 non-null   int64  
 8   fin       4360 non-null   int64  
 9   hisp      4360 non-null   int64  
 10  poorhlth  4360 non-null   int64  
 11  hours     4360 non-null   int64  
 12  manuf     4360 non-null   int64  
 13  married   4360 non-null   int64  
 14  min       4360 non-null   int64  
 15  nrthcen   4360 non-null   int64  
 16  nrtheast  4360 non-null   int64  
 17  occ1      4360 non-null   int64  
 18  occ2      4360 non-null   int64  
 19  occ3      4

次に`PanelData`オブジェクトに変換しデータの特徴を調べる。

In [7]:
PanelData(wagepan).shape

(44, 8, 545)

* 44: 変数の数
* 8: 期間数（年）
* 545：観察単位の数（人数）

---
実際に回帰式を書くことにする。使い方は`statsmodels`と似ている。
* `PanelOLS`モジュールの関数`.from_formula`を使い次のように引数を指定する。

$$\text{.from_formula}(\text{回帰式}, \text{データ})$$

* `EntityEffects`を加える。
* 定数項を入れたい場合は，`1`を回帰式に追加する。入れなければ定数項なしの推定となる。

* 以下では時間ダミー`C(year)`が入るので入れない。

In [8]:
formula_fe = 'lwage ~ married + union + expersq \
                      +d81+d82+d83+d84+d85+d86+d87 + EntityEffects'

* 固定効果モデルのインスタンスの作成

In [9]:
mod_fe = PanelOLS.from_formula(formula_fe, data=wagepan)

* `statsmodels`と同じように，そこから得た結果にメソッド`.fit()`を使い計算し結果が返される。

In [10]:
result_fe = mod_fe.fit()

＜結果の表示方法＞
1. `res_fe`を実行。
1. `res_fe`に関数`print()`を使うと見やすい。
1. `res_fe`には属性`summary`が用意されているが，表示方法1と同じ。
1. `summary`には属性`tables`があり，２つの表がリストとして格納されている。
    * `tables[0]`：検定統計量の表（`print()`を使うと見やすくなる）
    * `tables[1]`：係数の推定値やp値などの表（`print()`を使うと見やすくなる）

In [11]:
print(result_fe.summary.tables[1])

                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
married        0.0467     0.0183     2.5494     0.0108      0.0108      0.0826
union          0.0800     0.0193     4.1430     0.0000      0.0421      0.1179
expersq       -0.0052     0.0007    -7.3612     0.0000     -0.0066     -0.0038
d81            0.1512     0.0219     6.8883     0.0000      0.1082      0.1942
d82            0.2530     0.0244     10.360     0.0000      0.2051      0.3008
d83            0.3544     0.0292     12.121     0.0000      0.2971      0.4118
d84            0.4901     0.0362     13.529     0.0000      0.4191      0.5611
d85            0.6175     0.0452     13.648     0.0000      0.5288      0.7062
d86            0.7655     0.0561     13.638     0.0000      0.6555      0.8755
d87            0.9250     0.0688     13.450     0.00

（結果）
* `exper**2`の係数が負で統計的有意性が高いのは，賃金に対して経験の効果は低減することを示している。
* `married`の係数は正であり，優位性が全くないわけではない。賃金の既婚プレミアムと呼ばれるものである。
* `union`は労働組合の影響を示しているが，予測通りである。

$R^2$を表示してみる。

In [12]:
print(result_fe.summary.tables[0])

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.1806
Estimator:                   PanelOLS   R-squared (Between):              0.2386
No. Observations:                4360   R-squared (Within):               0.1806
Date:                Tue, Jan 13 2026   R-squared (Overall):              0.2361
Time:                        19:43:58   Log-likelihood                   -1324.8
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      83.851
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(10,3805)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             83.851
                            

＜この表にある$R^2$について＞
* $R^2$＝$R^2(\text{Within})$：(式３)を推定した際の$R^2$である（$\ddot{y}_i$が$\ddot{x}_i$によってどれだけ説明されたかを示す）。
* $R^2(\text{Between})$：(式２)を推定した際の$R^2$である（$\hat{y}_i$が$\hat{x}_i$によってどれだけ説明されたかを示す）。
* $R^2(\text{Overall})$：(式１)を推定した際の$R^2$である（${y}_i$が${x}_i$によってどれだけ説明されたかを示す）。

````{hint}
(式３)の回帰式には定数項はないため，上の計算で使った回帰式`formula_fe`には定数項を表す`1`は省かれている。(式３)と整合性が取れたコードとなっており，直観的にも分かりやすい。では，`formula_fe`に`1`を加えて次のように書き換えて計算するとどうなるのだろうか。
```
formula_fe = 'lwage ~ 1 + married + union + expersq \
                      +d81+d82+d83+d84+d85+d86+d87 + EntityEffects'
```
このコードを使うと定数項の推定値が計算されることになるが，(式３)とは整合性がないように見える。実際そうなのだが，`linearmodels`では，この場合の定数項の推定値は，(式３)の個体固有の因子である$a_i$の平均値を表す仕様になっている。`formula_fe`に`1+`が追加されても，他の変数の推定値などは変わらない。確かめてみよう！
````

#### TimeEffects

上の推定式では時間ダミー変数として使い，観察単位全てに共通な時間的な影響を捉えた。具体的には，インフレにより賃金は変化するが，その変化は全ての労働者には対して同じであり，その効果を時間ダミー変数が捉えている。それを**時間効果**と呼ぶ。このような時間ダミー変数を加えた理由は，時間効果が他の変数（例えば，`married`）の係数を「汚さない」ようにするためであり，よりピュアの効果を推定するためである。一方，`linearmodels`では，わざわざ時間ダミー変数を作らずとも`TimeEffects`を回帰式に追加することにより，時間効果を自動的に「排除」することができる。

In [13]:
formula_fe2 = 'lwage ~ married + union + expersq + TimeEffects + EntityEffects'
result_fe2 = PanelOLS.from_formula(formula_fe2, data=wagepan).fit()
print(result_fe2)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.0216
Estimator:                   PanelOLS   R-squared (Between):             -0.2717
No. Observations:                4360   R-squared (Within):              -0.4809
Date:                Tue, Jan 13 2026   R-squared (Overall):             -0.2808
Time:                        19:43:58   Log-likelihood                   -1324.8
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      27.959
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(3,3805)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             27.959
                            

`result_fe`と同じ結果を確認できる。$R^2$の値は少し変わっているが，これは時間ダミーを入れて計算している訳ではないためである。

# 練習問題

Use the data in RENTAL for this exercise. The data on rental prices and other variables for college
towns are for the years 1980 and 1990. The idea is to see whether a stronger presence of students
affects rental rates. The unobserved ettects model is

$$
\log(\text{rent}_{it})=\beta_0+\delta_0\text{y90}_t+\beta_1\log(\text{pop}_{it})
+\beta_2\log(\text{avginc}_{it})+\beta_3\text{pctstu}_{it}+a_{i}+u_{it}
$$

where `rent` is average rent (平均家賃), `pop` is city population（地域の人口）, `avginc` is average income（平均所得）, and `pctstu` is student population as a percentage of city population (学生の人口、％)（during the school year).

データを`df`に割り当てなさい。

In [14]:
df = wooldridge.data('rental')
wooldridge.data('rental', description=True)

name of dataset: rental
no of variables: 23
no of observations: 128

+----------+--------------------------------+
| variable | label                          |
+----------+--------------------------------+
| city     | city label, 1 to 64            |
| year     | 80 or 90                       |
| pop      | city population                |
| enroll   | # college students enrolled    |
| rent     | average rent                   |
| rnthsg   | renter occupied units          |
| tothsg   | occupied housing units         |
| avginc   | per capita income              |
| lenroll  | log(enroll)                    |
| lpop     | log(pop)                       |
| lrent    | log(rent)                      |
| ltothsg  | log(tothsg)                    |
| lrnthsg  | log(rnthsg)                    |
| lavginc  | log(avginc)                    |
| clenroll | change in lrent from 80 to 90  |
| clpop    | change in lpop                 |
| clrent   | change in lrent                |
| cltothsg 

In [15]:
df.head()

,city,year,pop,enroll,rent,rnthsg,tothsg,avginc,lenroll,lpop,...,lavginc,clenroll,clpop,clrent,cltothsg,clrnthsg,clavginc,pctstu,cpctstu,y90
0,1,80,75211.0,15303.0,197,13475.0,26167.0,11537.0,9.635804,11.228053,...,9.353314,NaN,NaN,NaN,NaN,NaN,NaN,20.346758,NaN,0
1,1,90,77759.0,18017.0,342,15660.0,29467.0,19568.0,9.799071,11.261370,...,9.881651,-15293.201172,0.033317,0.551607,0.118772,0.150273,0.528337,23.170309,2.823551,1
2,2,80,106743.0,22462.0,323,14580.0,37277.0,19841.0,10.019580,11.578179,...,9.895506,NaN,NaN,NaN,NaN,NaN,NaN,21.043066,NaN,0
3,2,90,141865.0,29769.0,496,26895.0,55540.0,31885.0,10.301223,11.862631,...,10.369891,-22451.699219,0.284451,0.428924,0.398727,0.612289,0.474385,20.984034,-0.059032,1
4,3,80,36608.0,11847.0,216,7026.0,13482.0,11455.0,9.379830,10.508022,...,9.346182,NaN,NaN,NaN,NaN,NaN,NaN,32.361778,NaN,0


## (i)
Estimate the equation by pooled OLS and report the results in standard form. What do you
make of the estimate on the 1990 dummy variable? What do you get for $\hat{\beta}_3$？

`statsmodels`を使う場合：定数項は自動で追加される

In [16]:
formula = 'lrent ~ y90 + lpop + lavginc + pctstu'
res = smf.ols(formula, data=df).fit()
print(res.summary(slim=True)) 

                            OLS Regression Results                            
Dep. Variable:                  lrent   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.857
No. Observations:                 128   F-statistic:                     190.9
Covariance Type:            nonrobust   Prob (F-statistic):           9.41e-52
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.5688      0.535     -1.063      0.290      -1.628       0.490
y90            0.2622      0.035      7.543      0.000       0.193       0.331
lpop           0.0407      0.023      1.807      0.073      -0.004       0.085
lavginc        0.5714      0.053     10.762      0.000       0.466       0.677
pctstu         0.0050      0.001      4.949      0.000       0.003       0.007

Notes:
[1] Standard Errors assume that the covarian

`linearmodels`を使う場合：定数項は手動で追加する必要がある

In [17]:
df = df.set_index(['city','year'])
df

pop   enroll  rent   rnthsg   tothsg   avginc    lenroll  \
city year                                                                  
1    80     75211.0  15303.0   197  13475.0  26167.0  11537.0   9.635804   
     90     77759.0  18017.0   342  15660.0  29467.0  19568.0   9.799071   
2    80    106743.0  22462.0   323  14580.0  37277.0  19841.0  10.019580   
     90    141865.0  29769.0   496  26895.0  55540.0  31885.0  10.301223   
3    80     36608.0  11847.0   216   7026.0  13482.0  11455.0   9.379830   
...             ...      ...   ...      ...      ...      ...        ...   
62   90     56856.0  10640.0   352   8980.0  21118.0  24735.0   9.272376   
63   80     48347.0   9051.0   220   8224.0  18085.0  13458.0   9.110631   
     90     51003.0   9961.0   344  10073.0  19970.0  21947.0   9.206432   
64   80    170616.0  37475.0   243  34108.0  66451.0  16510.0  10.531429   
     90    191262.0  44601.0   472  41029.0  77361.0  29420.0  10.705511   

                lpop     lrent    ltothsg  ...    lavginc      clenroll  \
city year                                  ...                            
1    80    11.228053  5.283204  10.172255  ...   9.353314           NaN   
     90    11.261370  5.834811  10.291026  ...   9.881651 -15293.201172   
2    80    11.578179  5.777652  10.526132  ...   9.895506           NaN   
     90    11.862631  6.206576  10.924859  ...  10.369891 -22451.699219   
3    80    10.508022  5.375278   9.509110  ...   9.346182           NaN   
...              ...       ...        ...  ...        ...           ...   
62   90    10.948277  5.863631   9.957881  ...  10.115974  -9600.727539   
63   80    10.786160  5.393628   9.802838  ...   9.507329           NaN   
     90    10.839640  5.840641   9.901986  ...   9.996386  -9041.793945   
64   80    12.047171  5.493062  11.104220  ...   9.711721           NaN   
     90    12.161400  6.156979  11.256238  ...  10.289430 -37464.292969   

              clpop    clrent  cltothsg  clrnthsg  clavginc     pctstu  \
city year                                                                
1    80         NaN       NaN       NaN       NaN       NaN  20.346758   
     90    0.033317  0.551607  0.118772  0.150273  0.528337  23.170309   
2    80         NaN       NaN       NaN       NaN       NaN  21.043066   
     90    0.284451  0.428924  0.398727  0.612289  0.474385  20.984034   
3    80         NaN       NaN       NaN       NaN       NaN  32.361778   
...             ...       ...       ...       ...       ...        ...   
62   90    0.098765  0.447531  0.140006  0.225234  0.537041  18.713943   
63   80         NaN       NaN       NaN       NaN       NaN  18.720913   
     90    0.053480  0.447014  0.099148  0.202803  0.489057  19.530224   
64   80         NaN       NaN       NaN       NaN       NaN  21.964529   
     90    0.114229  0.663918  0.152018  0.184747  0.577708  23.319321   

            cpctstu  y90  
city year                 
1    80         NaN    0  
     90    2.823551    1  
2    80         NaN    0  
     90   -0.059032    1  
3    80         NaN    0  
...             ...  ...  
62   90    0.057011    1  
63   80         NaN    0  
     90    0.809311    1  
64   80         NaN    0  
     90    1.354792    1  

[128 rows x 21 columns]

In [18]:
formula = 'lrent ~ 1 + y90 + lpop + lavginc + pctstu'
res = PooledOLS.from_formula(formula, data=df).fit()
# res = PooledOLS.from_formula(formula, data=df).fit(cov_type='unadjusted', debiased=True)
print(res.summary) 

                          PooledOLS Estimation Summary                          
Dep. Variable:                  lrent   R-squared:                        0.8613
Estimator:                  PooledOLS   R-squared (Between):              0.5557
No. Observations:                 128   R-squared (Within):               0.9694
Date:                Tue, Jan 13 2026   R-squared (Overall):              0.8613
Time:                        19:43:58   Log-likelihood                    86.161
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      190.92
Entities:                          64   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                   F(4,123)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             190.92
                            

## (ii)
Are the standard errors you report in part (i）valid? Explain.

### (iii)
Now, difference the equation and estimate by OLS. Compare your estimate of Bpetst With that from part（i). Does the relative size of the student population appear to affect rental prices？

In [19]:
formula = 'lrent ~ y90 + lpop + lavginc + pctstu'
res = FirstDifferenceOLS.from_formula(formula, data=df).fit()
# res = FirstDifferenceOLS.from_formula(formula, data=df).fit(cov_type='unadjusted', debiased=True)
print(res.summary)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:                  lrent   R-squared:                        0.9765
Estimator:         FirstDifferenceOLS   R-squared (Between):              0.9391
No. Observations:                  64   R-squared (Within):               0.9765
Date:                Tue, Jan 13 2026   R-squared (Overall):              0.9392
Time:                        19:43:58   Log-likelihood                    65.272
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      624.15
Entities:                          64   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                    F(4,60)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             624.15
                            

## (iv)
Estimate the model by fixed ettects to verity that you get identical estimates and standard errors to those in part（vi）.

In [20]:
formula = 'lrent ~ y90 + lpop + lavginc + pctstu + EntityEffects'
res = PanelOLS.from_formula(formula, data=df).fit()
# res = PanelOLS.from_formula(formula, data=df).fit(cov_type='unadjusted', debiased=True)
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lrent   R-squared:                        0.9765
Estimator:                   PanelOLS   R-squared (Between):              0.9391
No. Observations:                 128   R-squared (Within):               0.9765
Date:                Tue, Jan 13 2026   R-squared (Overall):              0.9392
Time:                        19:43:58   Log-likelihood                    219.27
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      624.15
Entities:                          64   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                    F(4,60)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             624.15
                            

In [21]:
formula = 'lrent ~ lpop + lavginc + pctstu + EntityEffects + TimeEffects'
res = PanelOLS.from_formula(formula, data=df).fit()
# res = PanelOLS.from_formula(formula, data=df).fit(cov_type='unadjusted', debiased=True)
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lrent   R-squared:                        0.3223
Estimator:                   PanelOLS   R-squared (Between):              0.9216
No. Observations:                 128   R-squared (Within):               0.5185
Date:                Tue, Jan 13 2026   R-squared (Overall):              0.9206
Time:                        19:43:58   Log-likelihood                    219.27
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      9.5099
Entities:                          64   P-value                           0.0000
Avg Obs:                       2.0000   Distribution:                    F(3,60)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             9.5099
                            

＜注意点＞
* 2期間パネルなため、一階差分推定と固定効果推定の推定値と推定値の標準誤差は同じになる。一般的な結果ではない。
* $\beta_3$の解釈の違い
    * 固定効果モデル：`pctstu`が高い地域は，平均的に賃貸料の対数がどう違うか
    * 一階差分モデル：`pctstu`が増えた時，平均的で賃貸料はどれだけ変わるか